# 02 - Pressure score

How hard is each bank being pushed to do something about its non-performing book.

This layer uses reported data only. It measures **capacity and constraint** --
what a balance sheet can afford to carry. It says nothing about management
intent, which is a separate signal and is handled in notebook 03. Conflating
the two is the most common weakness in screens like this.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)


In [2]:
from src import eba_data as E, pressure as P, config as C

panel  = E.build_bank_panel()
scored = P.add_pressure_score(panel)
C.PRESSURE_WEIGHTS

{'npl_ratio': 0.3,
 'net_npl_to_cet1': 0.3,
 'stage2_ratio': 0.2,
 'npl_stock': 0.2}

## Factor design

Four factors, percentile-ranked within the universe so the score is a relative
league table rather than an absolute claim:

- **NPL ratio** - how bad the book is relative to its own size
- **Unreserved NPLs / CET1** - the loss still to be taken, against the buffer
  that has to absorb it. This is the binding constraint in practice.
- **Stage 2 share** - forward pipeline of new NPL formation
- **Absolute NPL stock** (log) - is the position large enough to be investable
  at fund scale

They are deliberately non-overlapping. Coverage enters only through
`net_npl_to_cet1`, so a weakly provisioned bank is not penalised twice.

In [3]:
P.factor_table(scored, 15).round(1)

,bank,country,loans_total,loans_npl,npl_ratio,stage2_ratio,coverage_ratio,net_npl_to_cet1,cet1_ratio,pressure_score,transaction_ready
0,Deutsche Pfandbriefbank AG,DE,35972.3,2021.9,5.6,15.2,35.3,48.4,15.3,86.9,False
1,Landesbank Hessen-Thüringen Girozentrale,DE,125396.8,3919.0,3.1,15.2,29.6,29.9,16.3,85.8,False
2,Groupe BPCE,FR,998458.2,26279.3,2.6,13.7,39.4,21.6,16.3,83.8,False
3,"Banco Santander, S.A.",ES,1073587.2,35924.5,3.3,6.8,39.6,26.7,13.0,83.8,False
4,ATLANTIC LUX HOLDCO S.A R.L.,DE,33279.9,1477.0,4.4,13.5,21.5,39.7,21.9,83.5,False
5,Banca Monte dei Paschi di Siena S.p.A.,IT,83983.5,3064.4,3.6,11.3,46.0,18.4,19.6,80.0,True
6,Bpifrance,FR,58597.0,3932.8,6.7,17.9,23.0,12.8,25.0,79.2,False
7,Société générale S.A.,FR,506510.2,14006.9,2.8,8.2,44.0,14.9,13.5,74.2,False
8,Confédération Nationale du Crédit Mutuel,FR,750162.2,20134.1,2.7,7.5,46.3,14.8,19.8,71.5,True
9,BNP Paribas,FR,963618.1,26870.4,2.8,6.8,48.5,14.1,12.5,71.0,True


## Coverage is not pressure

A low coverage ratio is usually read as distress. It is better understood as
two opposing forces:

- a weakly provisioned bank faces **more** supervisory pressure to act, but
- crystallises a **larger** P&L hit when it sells, so it moves slower

A well-provisioned bank can transact near carrying value without taking further
pain. It is *readier*, not more desperate. So coverage is reported separately as
transaction readiness rather than folded into the pressure score.

In [4]:
ready = scored.groupby('transaction_ready')['bank'].count()
print("Banks able to sell near carrying value (coverage >= "
      f"{C.READINESS_COVERAGE_FLOOR:.0f}%):\n{ready.to_string()}")

P.country_summary(scored).round(1)

Banks able to sell near carrying value (coverage >= 45%):
transaction_ready
False    24
True     24


,banks,loans_total_eur_bn,npl_stock_eur_bn,npl_ratio,coverage_ratio,stage2_ratio,mean_pressure
country,,,,,,,
France,10,5032.1,123.3,2.4,44.4,9.7,56.6
Spain,10,2443.3,70.4,2.9,45.5,6.0,49.7
Germany,17,2488.8,46.1,1.9,35.0,13.5,48.8
Italy,11,1543.2,37.9,2.5,52.0,8.6,50.7


## The rotation

Germany shows the highest Stage 2 share and the lowest coverage of the four
systems. Italy -- the market most associated with NPL supply -- is now the best
provisioned.

The implication is not that Italy is uninteresting. It is that Italian banks are
*transaction-ready* while German banks are *accumulating a problem they have not
yet reserved against*. Those two states call for different origination
behaviour: price discipline in Italy, early relationship work in Germany.